# Data collection — master tables and anchor texts

**Author:** Fatima Zahra Rbaibi

This notebook documents how the inputs of the analysis were obtained through APIs: the two master tables, exported from the CKAN APIs of the open data portals of Porto and Águeda, and the three rewritten versions of the anchor texts, generated with the OpenAI API. Each row of a master table describes the **metadata** of one dataset (title, description, tags, categories, resources), not the content of its resource files.

> **Relation to the analysis.** The six analysis notebooks (`notebooks/porto/` and `notebooks/agueda/`) do not depend on this notebook: they read the immutable files in `data/`, whose SHA-256 hashes are recorded in `data/README.md`. A new run queries the portals as they are on the day of the run and therefore produces a **different snapshot** (another `exported_at` and, possibly, records added, removed or edited); a new generation of the anchors produces different texts. For this reason, everything this notebook produces is written to `outputs/data_collection/` and never over the files in `data/`; the code refuses to write in that folder.

| Section | Content |
|---|---|
| 1. Configuration | Paths, import of the collection code (`scripts/data_collection/`) and the generation switch |
| 2. Porto | Export of the master table (18 columns) |
| 3. Águeda | Export of the master table (14 columns) |
| 4. Comparison with the snapshots | Records kept, added and removed; adoption of a new export |
| 5. Anchor texts | Prompt, check of the preserved anchors and optional new generation (OpenAI API); API keys |

## 1. Configuration

The code that queries the APIs is in `scripts/data_collection/`, with one module per source (`porto.py`, `agueda.py`, `anchors.py`). This notebook only calls it, so that the same functions are documented in a single place and can be reused outside Jupyter. The repository root is located in the same way as in the analysis notebooks. The file name of each export records the date of the run.

> **Single editing point.** `GENERATE_ANCHORS` is the only switch of the notebook. With the default value (`False`), *Restart & Run All* queries only the two public portals and makes no call to the OpenAI API, which requires a personal key and has a cost.

In [ ]:
# ── Paths and imports ───────────────────────────────────────────────────────
import sys
from datetime import date
from pathlib import Path

import pandas as pd

PROJECT_DIR = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / "environment.yaml").is_file()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError("Open the project from the repository root or from the notebooks/ folder.")
sys.path.insert(0, str(PROJECT_DIR))
from scripts.data_collection import agueda, anchors, porto
from scripts.data_collection.snapshot import compare_with_snapshot

# True = new generation of the 18 anchors (requires OPENAI_API_KEY; has a cost)
GENERATE_ANCHORS = False

SNAPSHOT_DIR = PROJECT_DIR / "data"                       # read-only: inputs of the analysis
EXPORT_DIR = PROJECT_DIR / "outputs" / "data_collection"  # new exports
EXPORT_DATE = date.today().strftime("%Y%m%d")

PORTO_EXPORT = EXPORT_DIR / f"porto_tabela_mae_datasets_{EXPORT_DATE}.csv"
AGUEDA_EXPORT = EXPORT_DIR / f"agueda_tabela_mae_datasets_{EXPORT_DATE}.csv"
ANCHORS_EXPORT = EXPORT_DIR / f"ancoras_llm_3_versoes_{EXPORT_DATE}.csv"

print(f"Exports will be written to: {EXPORT_DIR.relative_to(PROJECT_DIR)}")

## 2. Porto

The Porto Open Data Portal (`opendata.porto.digital`) is queried through the `package_search` action, in pages of 100 records, without authentication. Each dataset gives one row with 18 columns, the same as in `data/porto_tabela_mae_datasets.csv`. The `exported_at` column records the moment of the export and is common to all the rows.

In [ ]:
# ── Export of the Porto master table ────────────────────────────────────────
df_porto = porto.export_master_table(PORTO_EXPORT)

print(f"\nPorto: {len(df_porto)} datasets exported to {PORTO_EXPORT.relative_to(PROJECT_DIR)}")
display(df_porto.head())

## 3. Águeda

The Águeda portal (`dadosabertos.cm-agueda.pt`) temporarily blocks sequences of fast requests (HTTP 403). The module therefore opens a session on the catalogue page, to obtain the cookies, queries pages of 25 records with a pause of 0.3 s between them and, after a 403, waits with progressive backoff before trying again. Each dataset gives one row with 14 columns, the same as in `data/agueda_tabela_mae_datasets.csv`. The `theme_category` column comes from the portal groups or, when these are empty, from the extras fields. As in the snapshot, there is no `exported_at` column: the date of the export is recorded in the file name.

In [ ]:
# ── Export of the Águeda master table ───────────────────────────────────────
df_agueda = agueda.export_master_table(AGUEDA_EXPORT)

print(f"Águeda: {len(df_agueda)} datasets exported to {AGUEDA_EXPORT.relative_to(PROJECT_DIR)}")
display(df_agueda.head())

## 4. Comparison with the snapshots used in the analysis

The table below compares each new export with the corresponding snapshot in `data/`: number of records, equality of the columns and dataset identifiers kept, added and removed. Differences are expected and describe the evolution of the portals since the snapshot was taken; they are not errors of the collection and do not change the results of the dissertation, which remain tied to the snapshots.

In [ ]:
# ── New export vs. snapshot in data/ ────────────────────────────────────────
comparison = pd.DataFrame([
    compare_with_snapshot(df_porto, SNAPSHOT_DIR / "porto_tabela_mae_datasets.csv", "dataset_id"),
    compare_with_snapshot(df_agueda, SNAPSHOT_DIR / "agueda_tabela_mae_datasets.csv", "id"),
])
display(comparison)

### Adoption of a new export

A new export replaces a snapshot only by a deliberate decision, recorded in the dissertation:

1. copy the file from `outputs/data_collection/` to `data/`, with the canonical name (`porto_tabela_mae_datasets.csv` or `agueda_tabela_mae_datasets.csv`);
2. update the description, the number of records and the SHA-256 hash in `data/README.md`;
3. run `python scripts/reproduce.py` and `python scripts/check_results.py` again.

The manifests (`run_manifest.json`) record the hashes of the inputs, so the comparisons between runs reject results produced with different data.

## 5. Anchor texts (OpenAI API)

The four versions of the anchors are inputs of the analysis, preserved in `data/`. The original version (`texto_base.csv`) contains the excerpts of the ENTI document for the six dimensions. The 50-, 100- and 150-word versions were generated on 12-06-2026 with the `gpt-5.4-mini` model, through the OpenAI Responses API, with one call per dimension and per version (18 calls), and saved in `ancoras_llm_3_versoes.csv` and in one file per version (`ancoras_v1_50_palavras.csv`, `ancoras_v2_100_palavras.csv`, `ancoras_v3_150_palavras.csv`). The code of that run is in `scripts/data_collection/anchors.py`, with the same prompt, model, word intervals and output token limit; the key is now read from `.env` or from an environment variable, never from the code.

`temperature`, `top_p` and the seed were not fixed, so a new run produces different texts. The generation step is therefore documented, but it is not reproducible word for word: the analysis is reproducible from the preserved files. A new generation is only carried out with `GENERATE_ANCHORS = True` and, if it were to replace the anchors in use, it would follow the same procedure as a new export (Section 4).

### 5.1 Prompt

The prompt is kept in Portuguese, as it was sent to the model. The cell below shows it for the first dimension in the 50-word version; for the other versions, only the approximate number of words and the interval change.

In [ ]:
# ── Prompt of the generation (first dimension, 50-word version) ─────────────
df_base = anchors.read_base_text(SNAPSHOT_DIR / "texto_base.csv")
first = df_base.iloc[0]
print(anchors.build_prompt(first["dimensao"], first["texto"], **anchors.ANCHOR_VERSIONS["v1_50_palavras"]))

The base text is read in latin-1, as in the original run, so that the prompts are identical to those sent to the model. With this encoding, the two symbols of the base text outside latin-1, «‰» in Smart Quality of Life and «€» in Smart Economy, reached the model as control characters; none of the 18 generated anchors contains them. The analysis notebooks read the same file in cp1252, which preserves both symbols in the original version of the anchors.

### 5.2 Length of the preserved anchors

The number of words of each preserved anchor is compared with the interval requested in the prompt. The check describes the output of the model; it does not change the files used in the analysis.

In [ ]:
# ── Preserved anchors vs. the word interval requested in the prompt ─────────
preserved_anchors = pd.read_csv(SNAPSHOT_DIR / "ancoras_llm_3_versoes.csv", sep=";", encoding="utf-8-sig")
word_check = anchors.word_count_check(preserved_anchors)
display(word_check)
print(f"Anchors outside the requested interval: {(~word_check['in_interval']).sum()} of {len(word_check)}")

**Analysis of the results.** Sixteen of the 18 preserved anchors respect the requested interval. The two exceptions belong to the 150-word version: Smart Governance, with 132 words, below the minimum of 140, and Smart Society, with 171, one word above the maximum of 170. The length was given to the model as an instruction and was not imposed afterwards, so the nominal lengths (50, 100 and 150 words) describe each version as a whole and not each anchor exactly. This characteristic of the inputs must be taken into account when the effect of the length of the anchors is interpreted.

### 5.3 New generation (optional)

In [ ]:
# ── New generation of the 18 anchors (only with GENERATE_ANCHORS = True) ───
if GENERATE_ANCHORS:
    df_new_anchors = anchors.export_anchors(SNAPSHOT_DIR / "texto_base.csv", ANCHORS_EXPORT)
    print(f"\n{len(df_new_anchors)} anchors written to {ANCHORS_EXPORT.relative_to(PROJECT_DIR)}")
    display(anchors.word_count_check(df_new_anchors))
else:
    print("GENERATE_ANCHORS = False: no call to the OpenAI API; the analysis uses the anchors in data/.")

### 5.4 API keys

The two CKAN APIs are public and need no key. The OpenAI key is read by `scripts.api_keys.get_api_key`, which looks first for an environment variable of the operating system and then for the file `.env` at the root of the repository, created by the user from `.env.example` and excluded from version control by `.gitignore`. The key is never written in a notebook, a script or an output cell, nor printed, even partially. A key that has been exposed, for example in a notebook shared with other people, must be revoked in the OpenAI dashboard and replaced.